# Create anndata for scVI

In [13]:
here::i_am("rna/scVI/scVI_get_anndata.ipynb")

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))


# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code



In [7]:
args <- list()
args$sce <-file.path(io$basedir,"processed/rna/SingleCellExperiment.rds")
args$metadataRNA <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
#args$metadataATAC <- file.path(io$basedir, '/results/atac/archR/qc/sample_metadata_after_qc.txt.gz')
args$outdir <- file.path(io$basedir,"results/scVI")

# I/O
dir.create(args$outdir, showWarnings=F, recursive=T)

In [15]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadataRNA) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE] %>%
  .[,exp:=str_replace_all(sample, opts$sample2exp)]

In [17]:
#########################
## Load RNA expression ##
#########################
  
sce <- load_SingleCellExperiment(
  file = args$sce, 
  normalise = FALSE, 
  cells = sample_metadata$cell, 
  remove_non_expressed_genes = TRUE
)

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

In [18]:
sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:pass_rnaQC, doublet_call”


AnnData object with n_obs × n_vars = 38346 × 23004
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'alias', 'day', 'genotype', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell', 'day_celltype', 'celltype_genotype', 'exp'
    var: 'name'

In [19]:
sce

class: SingleCellExperiment 
dim: 23004 38346 
metadata(0):
assays(1): counts
rownames(23004): Xkr4 Gm1992 ... CAAA01147332.1 AC149090.1
rowData names(0):
colnames(38346): 1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1
  1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1 ...
  rv_eo_deg_day4_dtag#TTTGTGTTCATTTGTC-1
  rv_eo_deg_day4_dtag#TTTGTTGGTACTTAGG-1
colData names(18): barcode sample ... celltype_genotype exp
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

In [11]:
args$outdir

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/scVI"